# Federated DTI prediction on Colab

Runs the centralized baseline, the local-only baseline or FedAvg (MPNN-CNN on DAVIS) on a Colab GPU.
Checkpoints and metrics are written to Google Drive after every epoch / round.

1. **Runtime → Change runtime type → GPU.**
2. Set the config cells below, then **Runtime → Run all**.
3. If Colab disconnects, reconnect and **Run all** again with the same `RUN_NAME`: training resumes from the last checkpoint on Drive.

To change hyperparameters, use a new `RUN_NAME` (the scripts refuse to resume a run with different settings). `EPOCHS` / `ROUNDS` may be increased to extend a finished run.

## 1. Config

In [ ]:
# ---- Repository ----
REPO_URL = "https://github.com/YOUR_USERNAME/federated-deeppurpose.git"  # <- set this
BRANCH = "master"
REPO_DIR = "/content/federated-deeppurpose"           # fixed clone location
REQUIREMENTS = f"{REPO_DIR}/requirements_colab.txt"

# ---- Output on Google Drive ----
DRIVE_DIR = "/content/drive/MyDrive/federated-deeppurpose"

# ---- Experiment ----
EXPERIMENT = "fedavg"            # "centralized" | "local" | "fedavg"
RUN_NAME = "fedavg_r30_e2_s42"   # one folder per run; reuse the name to resume
SEED = 42

In [ ]:
# ---- Hyperparameters ----
EPOCHS = 100        # centralized / local
ROUNDS = 30         # fedavg
LOCAL_EPOCHS = 2    # fedavg
BATCH_SIZE = 128
LR = 1e-3
NUM_WORKERS = 2
SUBSAMPLE = None    # e.g. 0.05 for a quick test run

## 2. Mount Drive and get the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)

Clones the repo to `/content/federated-deeppurpose` (or updates it if it is already there) and stops with an
error if that fails. For a private repository, use `https://<token>@github.com/...` as `REPO_URL`
(a GitHub personal access token with read access).

In [ ]:
import re
import subprocess

GIT = ["git", "-c", "credential.helper="]  # no credential helper prompts

def run(cmd):
    """Run a command, print its output (tokens masked) and return the result."""
    # never wait for a username/password prompt: fail immediately instead
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    for text in (r.stdout, r.stderr):
        if text:
            print(re.sub(r"//[^/@]+@", "//***@", text), end="")
    return r

if "YOUR_USERNAME" in REPO_URL:
    raise RuntimeError("Set REPO_URL in the config cell first.")

if os.path.isdir(f"{REPO_DIR}/.git"):
    action = "update"
    r = run([*GIT, "-C", REPO_DIR, "fetch", "origin", BRANCH])
    if r.returncode == 0:
        r = run([*GIT, "-C", REPO_DIR, "checkout", BRANCH])
    if r.returncode == 0:
        r = run([*GIT, "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH])
elif os.path.exists(REPO_DIR):
    raise RuntimeError(f"{REPO_DIR} exists but is not a git repository. Delete it (!rm -rf {REPO_DIR}) and rerun.")
else:
    action = "clone"
    r = run([*GIT, "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])

if r.returncode != 0 or not os.path.isfile(REQUIREMENTS):
    raise RuntimeError(
        f"git {action} failed (see the message above). Check that REPO_URL is correct, that branch "
        f"'{BRANCH}' exists on GitHub, and that a private repo URL includes a token.")

%cd {REPO_DIR}
branch = run([*GIT, "rev-parse", "--abbrev-ref", "HEAD"]).stdout.strip()
print(f"Repo dir: {REPO_DIR}")
print(f"Branch:   {branch}")
run([*GIT, "log", "-1", "--format=Commit:   %h %s"])

## 3. Install dependencies

Colab's own torch / numpy / pandas / scipy / scikit-learn / matplotlib / tensorboard are pinned with a constraints
file so pip cannot replace them. DeepPurpose is installed with `--no-deps` because its requirements pull in
`ax-platform` and `dgllife`, which MPNN-CNN does not use and which would reinstall torch.

In [ ]:
import sys

PROTECTED = {"torch", "torchvision", "torchaudio", "numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "tensorboard"}
CONSTRAINTS = os.path.join(os.path.dirname(REPO_DIR), "constraints.txt")  # /content/constraints.txt

if not os.path.isfile(REQUIREMENTS):
    raise RuntimeError(f"{REQUIREMENTS} not found. Run the clone cell above first.")

installed = subprocess.run([sys.executable, "-m", "pip", "list", "--format=freeze"],
                           capture_output=True, text=True).stdout
pins = [line for line in installed.splitlines() if line.split("==")[0].lower() in PROTECTED]
with open(CONSTRAINTS, "w") as f:
    f.write("\n".join(pins) + "\n")
print("Keeping Colab's versions:", ", ".join(pins))

for args in (["-r", REQUIREMENTS, "-c", CONSTRAINTS], ["--no-deps", "DeepPurpose==0.1.5"]):
    r = run([sys.executable, "-m", "pip", "install", "-q", *args])
    if r.returncode != 0:
        raise RuntimeError("pip install " + " ".join(args) + " failed (see the message above).")
print("Install finished.")

In [ ]:
import importlib
import torch, numpy, pandas

missing = []
for module in ["rdkit", "descriptastorus", "pandas_flavor", "lifelines", "subword_nmt", "prettytable",
               "wget", "tensorboard", "sklearn", "scipy", "matplotlib", "tqdm", "requests"]:
    try:
        importlib.import_module(module)
    except Exception as e:
        missing.append(f"  {module}: {e}")
if missing:
    raise RuntimeError("These packages failed to import:\n" + "\n".join(missing))

from DeepPurpose import DTI
print("torch", torch.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("CUDA available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
if not torch.cuda.is_available():
    print("WARNING: running on CPU. Runtime -> Change runtime type -> GPU")

Optional: check that the cached protein encoding matches DeepPurpose exactly on this machine (about a minute).

In [ ]:
# !python {REPO_DIR}/tests/verify_protein_cache.py --n 1024

## 4. Train

In [ ]:
import subprocess
import sys

OUT = f"{DRIVE_DIR}/outputs/{RUN_NAME}"
script = {
    "centralized": ["experiments/run_centralized.py", "--epochs", EPOCHS],
    "local": ["experiments/run_local.py", "--epochs", EPOCHS],
    "fedavg": ["experiments/run_federated.py", "--rounds", ROUNDS, "--local-epochs", LOCAL_EPOCHS],
}[EXPERIMENT]
args = ["--output-dir", OUT, "--seed", SEED, "--batch-size", BATCH_SIZE, "--lr", LR,
        "--num-workers", NUM_WORKERS, "--resume"]
if SUBSAMPLE:
    args += ["--subsample", SUBSAMPLE]

# the notebook's own Python (-u: unbuffered, so progress appears live)
cmd = [sys.executable, "-u", f"{REPO_DIR}/{script[0]}", *map(str, script[1:] + args)]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError(f"Training stopped with exit code {proc.returncode} (see the output above). "
                       "Rerun this cell to resume from the last checkpoint.")

## 5. Results

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

if EXPERIMENT == "local":
    display(pd.read_csv(f"{OUT}/local_summary.csv").round(4))
else:
    m = pd.read_csv(f"{OUT}/metrics.csv")
    step = m.columns[0]
    display(m.tail())
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for i, metric in enumerate(["mse", "ci"]):
        ax[i].plot(m[step], m[f"val_{metric}"], label="val")
        ax[i].plot(m[step], m[f"test_{metric}"], label="test")
        ax[i].set(xlabel=step, ylabel=metric.upper())
        ax[i].legend()
    plt.tight_layout()
    plt.show()

    final = f"{OUT}/final_metrics.json"
    if os.path.exists(final):
        with open(final) as f:
            result = json.load(f)
        print("Best model, global test:", json.dumps(result["test"]["all"], indent=2))